In [27]:
##### Converts raw rasters on livestock density into final varaiable needed 
# pixel and subnational (vector) scale
# variables: livestock density (heads per km2)

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats
from rasterio.features import geometry_mask

In [28]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# country geographies 
countries = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer="ADM_0")

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# raw raster
buffalo = f"{cd}/Data/Raw/FAO_gridded_livestock/GLW4-2020.D-DA.BFL.tif"
chicken = f"{cd}/Data/Raw/FAO_gridded_livestock/GLW4-2020.D-DA.CHK.tif"
cattle = f"{cd}/Data/Raw/FAO_gridded_livestock/GLW4-2020.D-DA.CTL.tif"
goats = f"{cd}/Data/Raw/FAO_gridded_livestock/GLW4-2020.D-DA.GTS.tif"
pigs = f"{cd}/Data/Raw/FAO_gridded_livestock/GLW4-2020.D-DA.PGS.tif"
sheep = f"{cd}/Data/Raw/FAO_gridded_livestock/GLW4-2020.D-DA.SHP.tif"

# Save paths
pixels_buffalo = f"{cd}/Data/Clean/Predictors/Rasters/livestock_density_buffalo_per_km2.tif"
pixels_chicken = f"{cd}/Data/Clean/Predictors/Rasters/livestock_density_chicken_per_km2.tif"
pixels_cattle = f"{cd}/Data/Clean/Predictors/Rasters/livestock_density_cattle_per_km2.tif"
pixels_goats = f"{cd}/Data/Clean/Predictors/Rasters/livestock_density_goats_per_km2.tif"
pixels_pigs = f"{cd}/Data/Clean/Predictors/Rasters/livestock_density_pigs_per_km2.tif"
pixels_sheep = f"{cd}/Data/Clean/Predictors/Rasters/livestock_density_sheep_per_km2.tif"

pixels_livestock = f"{cd}/Data/Clean/Predictors/Rasters/livestock_density_LU_per_km2.tif"

vectors = f"{cd}/Data/Clean/Predictors/Vectors/livestock_density_per_km2.csv"

In [29]:
#### Run resampling for pixel scale 

### PREP 

# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()
    ref_nodata    = ref.nodata
    ref_data      = ref.read(1)

# build reference valid mask
if ref_nodata is not None:
    ref_valid = ~np.isnan(ref_data) if np.isnan(ref_nodata) else (ref_data != ref_nodata)
else:
    ref_valid = ~np.isnan(ref_data)

# additional mask: reference is non-zero
ref_nonzero = ref_data != 0

# combine: only fill where ref is valid AND non-zero
should_fill = ref_valid & ref_nonzero

# set details for rasters
rasters_to_resample = {
    "buffalo": (buffalo, pixels_buffalo),
    "chicken": (chicken, pixels_chicken),
    "cattle": (cattle, pixels_cattle),
    "goats": (goats, pixels_goats),
    "pigs": (pigs, pixels_pigs),
    "sheep": (sheep, pixels_sheep),
}

### RESAMPLE AND FILL WITH ZEROS
for name, (src_path, out_path) in rasters_to_resample.items():
    with rasterio.open(src_path) as src:
        src_nodata = src.nodata

        dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst_array,
            dst_crs=dst_crs,
            dst_transform=dst_transform,
            resampling=Resampling.average,
            src_nodata=src_nodata,
            dst_nodata=np.nan,
        )

    # fill NaN only where reference is valid and non-zero
    dst_array[should_fill & np.isnan(dst_array)] = 0.0

    # save
    out_meta = dst_meta.copy()
    out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

    out_arr = dst_array.copy()
    out_arr[np.isnan(out_arr)] = -9999

    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(out_arr, 1)

    print(f"  [{name}] Saved → {out_path}")

  [buffalo] Saved → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/livestock_density_buffalo_per_km2.tif
  [chicken] Saved → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/livestock_density_chicken_per_km2.tif
  [cattle] Saved → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/livestock_density_cattle_per_km2.tif
  [goats] Saved → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/livestock_density_goats_per_km2.tif
  [pigs] Saved → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/livestock_density_pigs_per_km2.tif
  [sheep] Saved → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/livestock_density_sheep_per_km2.tif


In [30]:
#### Create combined Livestock Units raster

# FAO livestock unit weights (heads to LU conversion)
lu_weights = {
    "buffalo": 1.0,    # buffalo ≈ cattle
    "cattle": 1.0,
    "sheep": 0.1,
    "goats": 0.1,
    "pigs": 0.2,
    "chicken": 0.01,
}

# set rasters
output_rasters = {
    "buffalo": pixels_buffalo,
    "chicken": pixels_chicken,
    "cattle": pixels_cattle,
    "goats": pixels_goats,
    "pigs": pixels_pigs,
    "sheep": pixels_sheep,
}

# Initialize accumulator array
lu_array = np.zeros(dst_shape, dtype=np.float32)

# Sum each animal density raster
for name, out_path in output_rasters.items():
    with rasterio.open(out_path) as src:
        animal_data = src.read(1).astype(np.float32)
        animal_data[animal_data == -9999] = np.nan
        lu_array += np.nan_to_num(animal_data * lu_weights[name], nan=0.0)

# Apply reference mask: set to missing where ref is missing or 0
lu_output = lu_array.copy()
lu_output[~should_fill] = np.nan
lu_output[np.isnan(lu_output)] = -9999

# Save with same metadata as input rasters
lu_meta = dst_meta.copy()
lu_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

with rasterio.open(pixels_livestock, "w", **lu_meta) as dst:
    dst.write(lu_output, 1)

print(f"Livestock Units raster saved → {pixels_livestock}")

Livestock Units raster saved → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/livestock_density_LU_per_km2.tif


In [31]:
#### Run resampling for vector scale 

### Set-up 

# define animal rasters to process
rasters_to_process = {
    "buffalo_density_per_km2": pixels_buffalo,
    "chicken_density_per_km2": pixels_chicken,
    "cattle_density_per_km2": pixels_cattle,
    "goats_density_per_km2": pixels_goats,
    "pigs_density_per_km2": pixels_pigs,
    "sheep_density_per_km2": pixels_sheep,
}

# reproject GDF to match raster
with rasterio.open(pixels_livestock) as src:
    raster_crs = src.crs

gdf_proj = total_geo.to_crs(raster_crs)

# set final form 
result = total_geo[["PROJ_ID"]].copy()

### RESAMPLE ANIMAL DENSITIES
for col_name, rpath in rasters_to_process.items():
    with rasterio.open(rpath) as src:
        raster_data = src.read(1)
        raster_transform = src.transform
    
    densities = []
    for idx, row in gdf_proj.iterrows():
        # Create polygon mask
        poly_mask = geometry_mask(
            [row.geometry],
            transform=raster_transform,
            invert=True,
            out_shape=raster_data.shape
        )
        
        # Extract pixels within polygon that are non-zero and not missing
        valid_pixels = raster_data[poly_mask & (raster_data != -9999) & (raster_data > 0)]
        
        # Calculate mean, or 0 if no valid pixels
        if len(valid_pixels) > 0:
            mean_density = float(valid_pixels.mean())
        else:
            mean_density = 0.0
        
        densities.append(mean_density)
    
    result[col_name] = densities

### CALCULATE LIVESTOCK UNITS FROM ANIMAL COLUMNS
result["livestock_density_LU_per_km2"] = (
    result["buffalo_density_per_km2"] * lu_weights["buffalo"]
    + result["cattle_density_per_km2"] * lu_weights["cattle"]
    + result["sheep_density_per_km2"] * lu_weights["sheep"]
    + result["goats_density_per_km2"] * lu_weights["goats"]
    + result["pigs_density_per_km2"] * lu_weights["pigs"]
    + result["chicken_density_per_km2"] * lu_weights["chicken"]
)

### SAVE
result.to_csv(vectors, index=False)